In [ ]:
# 1. Download the repository files to Colab
!git clone https://github.com/andresmorenoviteri/injection_molding_machine_data_exp.git

# 2. Change the working directory to inside your repo
%cd injection_molding_machine_data_exp

In [ ]:
import pandas as pd

path = 'input_data.xlsx'
exp_data = pd.read_excel(path)
exp_data.head()

In [ ]:
exp_data.dtypes

In [ ]:
exp_data.shape

## Preprocessing chart data

In [ ]:
import os
import re
from collections import defaultdict
from functools import reduce
from typing import List, Dict, Any
import pandas as pd

# 1. NEW: Dynamically scan the 'chart_data' folder in the current directory
folder_name = "chart_data"

native_partitioned_input = {}
if os.path.exists(folder_name):
    for filename in os.listdir(folder_name):
        if filename.endswith(".txt"):
            file_stem = os.path.splitext(filename)[0]
            native_partitioned_input[file_stem] = filename

# =====================================================================
# Modified Functions (Updated base_path to look inside "chart_data")
# =====================================================================

def _group_partitions(partition_ids: List[str]) -> Dict[str, List[str]]:
    groups = defaultdict(list)
    pattern = re.compile(r"(.*_measurement_)\d+(_.*)")

    for p_id in partition_ids:
        match = pattern.match(p_id)
        if match:
            key = match.group(1) + match.group(2)
            groups[key].append(p_id)
    return groups


def _load_and_merge_group(partition_ids: List[str]) -> pd.DataFrame:
    df_list = []
    # FIX: Point directly to the folder next to your notebook
    base_path = "chart_data" 

    for p_id in sorted(partition_ids):
        try:
            df = pd.read_csv(
                f"{base_path}/{p_id}.txt",
                sep=";",
                skiprows=11,
                encoding="utf-16",
                engine="python",
            )
            if df.empty:
                continue

            df = df.iloc[:, :-1]
            df.columns = df.columns.str.strip()
            df_list.append(df)
        except Exception:
            continue

    if not df_list:
        return pd.DataFrame()

    return reduce(
        lambda left, right: pd.merge(left, right, on="time", how="outer"), df_list
    )


def preprocess_and_merge_charts(partitioned_input: Dict[str, Any]) -> pd.DataFrame:
    groups = _group_partitions(list(partitioned_input.keys()))
    merged_dfs = []
    
    for key, partition_ids in groups.items():
        df_merge = _load_and_merge_group(partition_ids)
        if not df_merge.empty:
            merged_dfs.append(df_merge)

    if not merged_dfs:
        return pd.DataFrame()

    all_dfs = []
    for idx, df in enumerate(merged_dfs):
        cleaned_df = df[df["time"] != "-start data-"].dropna(subset=["time"])
        temp_df = cleaned_df.copy()
        temp_df["id"] = idx + 1
        all_dfs.append(temp_df)

    chart_data = pd.concat(all_dfs, ignore_index=True)
    chart_data = chart_data.apply(pd.to_numeric, errors="coerce")
    chart_data = chart_data.sort_values(by=["id", "time"]).reset_index(drop=True)

    return chart_data

# =====================================================================
# 2. Run the processing pipeline
# =====================================================================
chart_data = preprocess_and_merge_charts(native_partitioned_input)
chart_data.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

def plot_chart_data(chart_df: pd.DataFrame):

    # Initialize a new figure with specific dimensions
    plt.figure(figsize=(10,6))

    # Get a list of all column names except for "time" to identify the sensor parameters
    params = chart_df.columns.drop(["time", "id"])

    # scale data only for plot
    # scaler = MinMaxScaler()
    # scaled_chart_data = pd.DataFrame(scaler.fit_transform(chart_df[params]), columns=params)
    # scaled_chart_data['time'] = chart_df['time']
    # scaled_chart_data['id'] = chart_df['id']
    scaled_chart_data = chart_data

    # Iterate through the parameter list to plot each sensor's data
    for idx, param in enumerate(params):
        # choose the parameter index to plot
        if idx == 1:
            for id in range(75, 77):
                # Create a line plot for the current parameter against the time axis
                sns.lineplot(data=scaled_chart_data[scaled_chart_data['id'] == id], x='time', y=param, label=param)

    # # Set the descriptive title and axis labels for the chart
    # plt.title("Sensor Parameters")
    # plt.xlabel("time")
    # plt.ylabel("Param value")

    # # Place the legend in the bottom right corner to avoid overlapping with data lines
    # plt.legend(loc="lower right")

            # Render the final plot
            plt.show()

In [ ]:
plot_chart_data(chart_data)